In [ ]:
from pathlib import Path
import os
import sys

import mne

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib qt

Channels marked as bad:
none
Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


c:\Repositorios\msc-eeg-tms-pipeline\.venv\Lib\site-packages\ipykernel\eventloops.py:158: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  el.exec() if hasattr(el, "exec") else el.exec_()


Dropped 7 epochs: 22, 23, 24, 25, 26, 27, 28
The following epochs were marked as bad and are dropped:
[np.int64(465), np.int64(501), np.int64(519), np.int64(573), np.int64(600), np.int64(645), np.int64(690)]
Channels marked as bad:
none
Dropped 1 epoch: 2
The following epochs were marked as bad and are dropped:
[np.int64(141)]
Channels marked as bad:
none


In [ ]:
from utils.decode_trigger import decode_8bit_trigger, convert_dict_trigger

In [4]:
raw_data = mne.io.read_raw_bdf(r"data/raw/V1.bdf", preload=True)

Extracting BDF parameters from data/raw/V1.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 12594999  =      0.000 ...  2519.000 secs...


In [5]:
events, event_id = mne.events_from_annotations(raw_data)
event_id = raw_data.event_id = convert_dict_trigger(event_id, decode_8bit_trigger)

Used Annotations descriptions: [np.str_('8Bit 1'), np.str_('8Bit 10'), np.str_('8Bit 2'), np.str_('8Bit 3'), np.str_('8Bit 4'), np.str_('8Bit 5'), np.str_('Stimulus A')]


In [6]:
emg_ch_names = ["EMG L", "EMG R"]
raw_emg = raw_data.copy().pick(emg_ch_names)
raw_emg.set_channel_types({ch: "emg" for ch in emg_ch_names})
raw_emg.load_data()

emg_picks = mne.pick_types(raw_emg.info, emg=True)
emg_picks


array([0, 1])

In [8]:
raw_emg.filter(
    1,
    50,
    method="iir",
    iir_params={"order": 4, "ftype": "butter"},
    picks=emg_picks,
)
raw_emg.notch_filter(
    60,
    method="iir",
    iir_params={"order": 4, "ftype": "butter"},
    picks=emg_picks,
)

No data channels found. The highpass and lowpass values in the measurement info will not be updated.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 50 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 1.00, 50.00 Hz: -6.02, -6.02 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

IIR filter parameters
---------------------
Butterworth bandstop zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 59.35, 60.65 Hz: -6.02, -6.02 dB



<RawBDF | V1.bdf, 2 x 12595000 (2519.0 s), ~192.2 MiB, data loaded>

In [13]:
epochs_left = mne.Epochs(
    raw_emg,
    events,
    event_id=event_id["task_left"],
    tmin=0,
    tmax=8,
    baseline=None,
    preload=True,
    picks=emg_picks,
)

epochs_right= mne.Epochs(
    raw_emg,
    events,
    event_id=event_id["task_right"],
    tmin=0,
    tmax=8,
    baseline=None,
    preload=True,
    picks=emg_picks,
)

epochs_bilateral = mne.Epochs(
    raw_emg,
    events,
    event_id=event_id["task_bilateral"],
    tmin=0,
    tmax=8,
    baseline=None,
    preload=True,
    picks=emg_picks,
)


Not setting metadata
36 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 36 events and 40001 original time points ...


0 bad epochs dropped
Not setting metadata
27 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 27 events and 40001 original time points ...
0 bad epochs dropped
Not setting metadata
37 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 37 events and 40001 original time points ...
0 bad epochs dropped


In [27]:
import numpy as np

In [25]:
def get_peak_frequencies(epochs, ch_name, fmin=0.5, fmax=5.0):
    """
    Calcula o envelope, faz o PSD por trial e retorna a frequência de pico.
    """
    # 1. Extrair dados (n_epochs, n_channels, n_times)
    # Pegamos apenas o canal de interesse
    data = epochs.copy().pick(ch_name).get_data() 
    sfreq = epochs.info['sfreq']
    
    # 2. Obter o envelope: Retificação + Filtro Passa-Baixa (ex: 10Hz)
    # Usamos o mne.filter para filtrar os dados brutos das épocas
    rectified_data = np.abs(data)
    envelope = mne.filter.filter_data(rectified_data, sfreq, l_freq=None, h_freq=10.0, verbose=False)
    
    # 3. Calcular PSD para cada trial (epoch) individualmente
    # Usamos o método multitaper ou welch
    psd_dict = mne.time_frequency.psd_array_multitaper(
        envelope, sfreq, fmin=fmin, fmax=fmax, verbose=False
    )
    psds, freqs = psd_dict[0], psd_dict[1]
    
    # 4. Encontrar a frequência de pico para cada trial
    # psds tem formato (n_epochs, 1, n_freqs)
    peak_freqs = []
    for i in range(psds.shape[0]):
        idx_max = np.argmax(psds[i, 0, :])
        peak_freqs.append(freqs[idx_max])
        
    return peak_freqs

In [36]:
results = []

# Processar Condição Left (apenas canal Left)
peaks = get_peak_frequencies(epochs_left, "EMG L")
for p in peaks[-10:]: results.append({"Frequência": p, "Condição": "Left", "Canal": "EMG L"})

# Processar Condição Right (apenas canal Right)
peaks = get_peak_frequencies(epochs_right, "EMG R")
for p in peaks[-10:]: results.append({"Frequência": p, "Condição": "Right", "Canal": "EMG R"})

# Processar Condição Bilateral (ambos os canais)
peaks_l = get_peak_frequencies(epochs_bilateral, "EMG L")
peaks_r = get_peak_frequencies(epochs_bilateral, "EMG R")
for p in peaks_l[-10:]: results.append({"Frequência": p, "Condição": "Bilateral", "Canal": "EMG L"})
for p in peaks_r[-10:]: results.append({"Frequência": p, "Condição": "Bilateral", "Canal": "EMG R"})

In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [37]:
df_results = pd.DataFrame(results)

plt.figure(figsize=(10, 6))
sns.boxplot(x="Condição", y="Frequência", hue="Canal", data=df_results, palette="Set2")
sns.stripplot(x="Condição", y="Frequência", hue="Canal", data=df_results, 
              dodge=True, alpha=0.5, color="black", legend=False)

plt.title("Distribuição das Frequências de Ritmo por Trial")
plt.ylabel("Frequência de Pico (Hz)")
plt.xlabel("Condição Experimental")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

C:\Users\marci\AppData\Local\Temp\ipykernel_24444\172432838.py:5: FutureWarning: 

Setting a gradient palette using color= is deprecated and will be removed in v0.14.0. Set `palette='dark:black'` for the same effect.

  sns.stripplot(x="Condição", y="Frequência", hue="Canal", data=df_results,


In [34]:
df_results

,Frequência,Condição,Canal
0,0.624984,Left,EMG L
1,0.624984,Left,EMG L
2,0.624984,Left,EMG L
3,0.624984,Left,EMG L
4,0.624984,Left,EMG L
...,...,...,...
132,2.749931,Bilateral,EMG R
133,2.624934,Bilateral,EMG R
134,0.624984,Bilateral,EMG R
135,2.874928,Bilateral,EMG R
